# 05 — Supervised Transfer Learning with MobileNetV2

**Thesis:** *A Mobile-Integrated Deep Learning System for Rapid Screening of Respiratory Diseases Using Cough Sound Analysis*  
**Student:** Nkenfa Nkombong Brandon (CT24P016) — University of Buea, College of Technology  
**Supervisor:** Dr. Tchapga Tchito Christian

## Why this notebook exists

The earlier baseline (notebooks 03 and 04) trained a small custom 2D CNN **from scratch** on log-mel spectrograms.
The supervisor asked us to apply the principles of transfer learning, pretrained models, layer replacement,
freezing early layers, fine-tuning upper layers, and comparing different algorithms.

This notebook documents the upgrade:

> **Supervised transfer learning using a pretrained MobileNetV2 backbone on log-mel spectrogram representations of cough audio, with balanced class weights to address COVID/HEALTHY class imbalance.**

The previous baseline is **not deleted** — it is preserved as one of several algorithms compared in notebook 06.

## 1. Dataset used

| Split | COVID | HEALTHY_OR_NONTARGET | Total |
|---|---|---|---|
| Train | 991 | 3317 | 4308 |
| Val | 204 | 550 | 754 |
| Test | 204 | 550 | 754 |

Feature shape per sample: **(64 mel bands × 256 time frames × 1 channel)** — log-mel spectrogram, min-max scaled to [0, 1].

Sources: **Coswara** (5,015 samples) and **COUGHVID** (13,634 samples), Zenodo open access.
Labels remapped from the 3-class schema (TB=0, COVID=1, HEALTHY=2) to binary (COVID=0, HEALTHY=1).
No TB samples are present in the current splits; the remap step drops them if they appear.

## 2. From-scratch supervised learning vs. transfer learning

| Aspect | Baseline CNN (notebooks 03–04) | MobileNetV2 transfer learning (this notebook) |
|---|---|---|
| Initial weights | random | pretrained on 1.2 M ImageNet images |
| Feature extractor | learned from scratch on ~4,300 cough clips | reuses low-level edge/texture filters from ImageNet |
| Risk on small imbalanced datasets | underfits or overfits easily | better-conditioned starting point |
| Training cost | full network updated | only head (phase 1) + top 30 layers (phase 2) |
| Mobile deployability | custom architecture, TFLite export failed | MobileNetV2 is TFLite-native |

Transfer learning is appropriate here because the cough corpus (~4,300 training samples)
is small and class-imbalanced (1:3.3 COVID:HEALTHY). ImageNet-pretrained convolutional
filters encode generic edge, texture, and frequency-band detectors that transfer well
to spectrogram inputs.

## 3. Why log-mel spectrograms can be fed into an image CNN

A log-mel spectrogram is a 2D matrix where rows index mel-frequency bands, columns index
time frames, and values are log-magnitude energies — structurally identical to a
single-channel grayscale image:

- **vertical structure** ↔ frequency content (vowel formants, voiced/unvoiced regions)
- **horizontal structure** ↔ temporal envelope (cough onset, burst, decay)
- **local 2D patterns** ↔ characteristic cough textures that differ across diseases

MobileNetV2 expects `(224, 224, 3)`. The pipeline produces `(64, 256, 1)`.
Three preprocessing layers are embedded **inside the model graph** so the saved
`.keras` / `.tflite` file is fully self-contained:

1. `Resizing(224, 224)` — bilinear up-sample.
2. `Concatenate([x, x, x])` — replicate single channel to 3 channels.
3. `Rescaling(255.0)` then `Rescaling(1/127.5, offset=-1)` — maps [0,1] → [-1,1].

> **Note on the Lambda layer issue:** The first training attempt used
> `tf.keras.layers.Lambda(mobilenet_v2.preprocess_input)`. Keras 3 cannot
> serialize anonymous Python functions, so the model failed to reload for
> TFLite export. Replacing it with two `Rescaling` layers (mathematically
> identical) fixes serialization cleanly.

## 4. Model architecture

```
Input: (64, 256, 1)  ← log-mel spectrogram
  │
  ├─ Resizing(224, 224)           ← bilinear upsample
  ├─ Concatenate([x,x,x])         ← grayscale → RGB
  ├─ Rescaling(255.0)             ← [0,1] → [0,255]
  ├─ Rescaling(1/127.5, -1)       ← [0,255] → [-1,1]  (MobileNetV2 norm)
  │
  ├─ MobileNetV2(include_top=False, weights='imagenet')  ← 154 layers, frozen phase 1
  │    └─ top 30 layers unfrozen in phase 2 (BN kept frozen)
  │
  ├─ GlobalAveragePooling2D
  ├─ Dropout(0.3)
  ├─ Dense(128, ReLU)
  ├─ Dropout(0.3)
  └─ Dense(2, softmax)            ← [P(COVID), P(HEALTHY)]
```

### Class weights

To handle the 1:3.3 COVID:HEALTHY imbalance, `sklearn.utils.class_weight.compute_class_weight('balanced')`
is applied during training:

| Class | Weight |
|---|---|
| COVID (0) | ~2.17 |
| HEALTHY (1) | ~0.65 |

This causes the loss to penalise missed COVID cases ~3.3× more than missed HEALTHY cases,
aligning the training objective with the screening goal.

## 5. Setup and data loading

In [ ]:
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# Make sure project root is on the path when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
set_seed(42)

FEATURES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'features'
CHECKPOINTS_DIR = PROJECT_ROOT / 'models' / 'checkpoints'
EXPORTED_DIR   = PROJECT_ROOT / 'models' / 'exported'
TABLES_DIR     = PROJECT_ROOT / 'reports' / 'tables'

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
EXPORTED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f'TensorFlow {tf.__version__}')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Label remap: original schema COVID=1, HEALTHY=2 → binary COVID=0, HEALTHY=1
ORIGINAL_TO_BINARY = {1: 0, 2: 1}
LABEL_NAMES = ['COVID', 'HEALTHY']

def load_split(name):
    x = np.load(FEATURES_DIR / f'X_{name}.npy')
    y = np.load(FEATURES_DIR / f'y_{name}.npy')
    keep = np.isin(y, [1, 2])          # drop TB rows if any
    x = x[keep].astype(np.float32)
    y = np.array([ORIGINAL_TO_BINARY[int(v)] for v in y[keep]], dtype=np.int64)
    return x, y

x_train, y_train = load_split('train')
x_val,   y_val   = load_split('val')
x_test,  y_test  = load_split('test')

print(f'Train: {x_train.shape}  COVID={np.sum(y_train==0)}  HEALTHY={np.sum(y_train==1)}')
print(f'Val:   {x_val.shape}   COVID={np.sum(y_val==0)}   HEALTHY={np.sum(y_val==1)}')
print(f'Test:  {x_test.shape}   COVID={np.sum(y_test==0)}   HEALTHY={np.sum(y_test==1)}')

## 6. Balanced class weights

In [ ]:
weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
class_weights = {0: float(weights_arr[0]), 1: float(weights_arr[1])}

print(f"COVID  weight : {class_weights[0]:.4f}")
print(f"HEALTHY weight: {class_weights[1]:.4f}")
print(f"→ COVID errors penalised {class_weights[0]/class_weights[1]:.1f}× more")

## 7. Build model

In [ ]:
inputs = tf.keras.Input(shape=x_train.shape[1:], name='logmel_input')

# --- In-graph preprocessing: (64,256,1) → MobileNetV2-ready (224,224,3) in [-1,1] ---
x = tf.keras.layers.Resizing(224, 224, interpolation='bilinear', name='resize')(inputs)
x = tf.keras.layers.Concatenate(name='to_rgb')([x, x, x])
# Two Rescaling layers replace Lambda(preprocess_input) for Keras 3 compatibility.
# Lambda layers wrapping Python functions cannot be serialized in Keras 3.
x = tf.keras.layers.Rescaling(255.0, name='scale_up')(x)
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1, name='mobilenet_norm')(x)

# --- Pretrained MobileNetV2 backbone (frozen for phase 1) ---
base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False
x = base(x, training=False)

# --- Classifier head ---
x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
x = tf.keras.layers.Dropout(0.3, name='drop1')(x)
x = tf.keras.layers.Dense(128, activation='relu', name='dense')(x)
x = tf.keras.layers.Dropout(0.3, name='drop2')(x)
out = tf.keras.layers.Dense(2, activation='softmax', name='class_probs')(x)

model = tf.keras.Model(inputs=inputs, outputs=out, name='mobilenetv2_screening')
model.summary()

## 8. Two-phase training

### Phase 1 — head training (backbone frozen)

In [ ]:
CKPT_PATH = str(CHECKPOINTS_DIR / 'mobilenetv2_screening.keras')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        CKPT_PATH, monitor='val_loss', save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=4, restore_best_weights=True
    ),
]

print('=== PHASE 1: head training (backbone frozen) ===')
history_phase1 = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    batch_size=16,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

### Phase 2 — fine-tuning (top 30 backbone layers, BN frozen)

In [ ]:
# Unfreeze top 30 backbone layers; keep BatchNormalization frozen for stability
base.trainable = True
cutoff = max(len(base.layers) - 30, 0)
for i, layer in enumerate(base.layers):
    if i < cutoff or isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in base.layers if l.trainable)
print(f'Unfrozen backbone layers: {trainable_count} (BN excluded)')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),   # 100x lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('=== PHASE 2: fine-tuning (top 30 layers, LR=1e-5) ===')
history_phase2 = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    batch_size=16,
    epochs=5,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

## 9. Evaluation on held-out test set

In [ ]:
import json
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)

COVID_THRESHOLD = 0.35  # tuned on val set to maximise COVID recall

probs = model.predict(x_test, batch_size=16, verbose=0)
covid_probs   = probs[:, 0]
default_pred  = np.argmax(probs, axis=1)
threshold_pred = np.where(covid_probs >= COVID_THRESHOLD, 0, 1)

def metrics_block(y_pred, label):
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0,1], zero_division=0)
    acc = accuracy_score(y_test, y_pred)
    cm  = confusion_matrix(y_test, y_pred, labels=[0,1])
    print(f'\n--- {label} ---')
    print(f'Accuracy        : {acc:.4f}')
    print(f'COVID precision : {p[0]:.4f}   recall: {r[0]:.4f}   F1: {f1[0]:.4f}')
    print(f'HEALTHY precision: {p[1]:.4f}   recall: {r[1]:.4f}   F1: {f1[1]:.4f}')
    print(f'Confusion matrix (COVID=0, HEALTHY=1):')
    print(f'  {cm}')
    return dict(accuracy=float(acc),
                covid_precision=float(p[0]), covid_recall=float(r[0]), covid_f1=float(f1[0]),
                healthy_precision=float(p[1]), healthy_recall=float(r[1]), healthy_f1=float(f1[1]),
                confusion_matrix=cm.tolist())

m_default   = metrics_block(default_pred,   'Default argmax (threshold=0.5)')
m_threshold = metrics_block(threshold_pred, f'Threshold-tuned (COVID prob >= {COVID_THRESHOLD})')

results = {
    'model_name': 'mobilenetv2_screening',
    'algorithm': 'Supervised transfer learning (MobileNetV2 backbone, class weights, threshold 0.35)',
    'threshold_covid': COVID_THRESHOLD,
    'class_weights': {'COVID': round(class_weights[0], 4), 'HEALTHY': round(class_weights[1], 4)},
    'default_argmax': m_default,
    'threshold_tuned': m_threshold,
    'num_test_samples': int(len(y_test)),
}

out_path = TABLES_DIR / 'mobilenetv2_screening_metrics.json'
out_path.write_text(json.dumps(results, indent=2))
print(f'\nMetrics saved to {out_path}')

## 10. TFLite export

### What failed on the first attempt
- The first trained model used `tf.keras.layers.Lambda(mobilenet_v2.preprocess_input)`.
- Keras 3 cannot serialize anonymous Python functions, so `load_model()` raised
  `TypeError: Could not locate function 'preprocess_input'`.
- The TFLite converter then crashed with `LLVM ERROR: Failed to infer result type(s)`.

### Fix applied
- Replaced `Lambda` with two `Rescaling` layers (mathematically identical, fully serializable).
- Export path: **SavedModel first, then TFLite converter from SavedModel** (more stable in TF 2.16).

In [ ]:
saved_model_path = str(EXPORTED_DIR / 'mobilenetv2_screening_savedmodel')
tflite_path      = EXPORTED_DIR / 'mobilenetv2_screening.tflite'

print('=== Saving as SavedModel ===')
model.export(saved_model_path)
print(f'SavedModel written to {saved_model_path}')

print('\n=== Converting SavedModel → TFLite ===')
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
tflite_model = converter.convert()
tflite_path.write_bytes(tflite_model)
print(f'Export SUCCESS: {tflite_path} ({len(tflite_model)/1024/1024:.2f} MB)')

print('\n=== Verifying TFLite interpreter ===')
interp = tf.lite.Interpreter(model_path=str(tflite_path))
interp.allocate_tensors()
inp_detail = interp.get_input_details()[0]
out_detail = interp.get_output_details()[0]
print(f"Input : name={inp_detail['name']}  shape={inp_detail['shape']}  dtype={inp_detail['dtype']}")
print(f"Output: name={out_detail['name']}  shape={out_detail['shape']}  dtype={out_detail['dtype']}")

# Run one sample to confirm end-to-end correctness
interp.set_tensor(inp_detail['index'], x_test[:1])
interp.invoke()
result = interp.get_tensor(out_detail['index'])
print(f'Sample inference: COVID_prob={result[0][0]:.4f}  HEALTHY_prob={result[0][1]:.4f}')
print('TFLite model verified OK')

## 11. Results on the held-out test split

### Comparison with previous best baseline

| Metric | Threshold-tuned baseline CNN | MobileNetV2 (weighted, threshold 0.35) | Change |
|---|---|---|---|
| Accuracy | 0.6724 | 0.6711 | −0.0013 |
| COVID precision | 0.4201 | 0.4197 | −0.0004 |
| **COVID recall** | 0.5539 | **0.5637** | **+0.0098** |
| **COVID F1** | 0.4778 | **0.4812** | **+0.0034** |
| Healthy recall | 0.7164 | 0.7109 | −0.0055 |

The MobileNetV2 transfer-learning model improves COVID recall and F1 — the two
metrics that matter most for a screening tool — with negligible trade-off on
the other metrics. This constitutes a genuine (if modest) improvement over the
custom CNN baseline, despite training on CPU with no GPU acceleration.

## 12. Mobile deployment path

The `lively/` React Native / Expo app contains:

- `lively/assets/model/mobilenetv2_screening.tflite` — deployed model (9.1 MB).
- `lively/src/ml/tfliteModel.ts` — TFLite inference wrapper using `react-native-fast-tflite`.
- `lively/src/ml/audioPreprocess.ts` — TypeScript port of the spectrogram pipeline.
- `lively/src/ml/offlineInference.ts` — offline inference coordinator.

Because preprocessing (resize, channel replication, normalization) is **inside**
the TFLite graph, the TypeScript code only needs to produce the raw `(64, 256, 1)`
float32 spectrogram — no changes to the normalization step are needed on the app side.

The app calls `runOfflineScreeningInference(uri)` which applies the 0.35 threshold
and returns `{ covidProb, healthyProb, covidFlag }`.